In [2]:
import re
from enum import Enum
from dataclasses import dataclass
from typing import Optional  # Toujours nécessaire pour Optional

# Équivalents des énumérations TypeScript
class Genre(Enum):
    HOMME = "HOMME"
    FEMME = "FEMME"
    # Ajoutez les autres valeurs selon votre besoin

class CodeDossier(Enum):
    TRA = "TRA"
    LIV = "LIV"
    # Ajoutez les autres valeurs selon votre besoin

class Phase(Enum):
    DUPLI = "DUPLI"
    PROD = "PROD"
    # Ajoutez les autres valeurs selon votre besoin

# Utilisation de dataclass pour PathInfos (Python 3.7+)
@dataclass
class PathInfos:
    trimestre: Optional[str] = None
    genre: Optional[Genre] = None
    collection: Optional[str] = None
    animation: Optional[str] = None
    code_dossier: Optional[CodeDossier] = None
    phase: Optional[Phase] = None
    segment: Optional[str] = None
    produit: Optional[str] = None
    dimensions: Optional[str] = None
    version: Optional[int] = None
    date: Optional[str] = None

def extract_basename_and_extension(filename: str) -> tuple[str, str]:
    """
    Extrait le nom de base et l'extension d'un nom de fichier.

    Args:
        filename: Le nom du fichier

    Returns:
        Un tuple contenant le nom de base et l'extension
    """
    parts = filename.split('.')
    if len(parts) == 1:
        return filename, ""
    extension = parts[-1]
    basename = '.'.join(parts[:-1])
    return basename, extension

def parse_path(path: str) -> tuple[PathInfos, bool, list[str]]:
    """
    Analyse un chemin de fichier et en extrait des informations structurées basées sur les conventions de nommage des dossiers/fichiers en vigueur.

    La fonction suppose que la chaîne de caractères représentant le chemin, et en particulier le nom du fichier, suit un format spécifique.
    Chaque segment du chemin est analysé et les informations sur le chemin sont renvoyées dans un objet structuré.

    Args:
        path: Le chemin de fichier à analyser.

    Returns:
        Un 3-uple contenant :
          - Un objet `PathInfos` rempli avec les détails extraits du chemin. Si certains éléments ne peuvent être trouvés,
            ils resteront nuls dans l'objet PathInfos.
          - Un booléen indiquant si le chemin de fichier est valide : il vaudra `False` si le chemin, et en particulier le nom de fichier,
            ne respecte pas le format attendu ; il vaudra `True` sinon.
          - Une liste de chaînes de caractères représentant les erreurs rencontrées lors de l'extraction des informations.
    """

    # On part d'un objet qui ne contient aucune information
    path_infos = PathInfos()

    # Chaîne vide
    if not path:
        return path_infos, False, ["Le chemin est vide."]

    # Les noms de dossiers sont séparés par le caractère `/` ; on gère le cas où plusieurs `/` se suivent
    parts = [part for part in path.split('/') if part]

    i = 0
    # On trouve l'index du premier dossier dont le nom ressemble à un trimestre (exemple : `24-Q1`)
    while i < len(parts) and not re.match(r'^\d{2}-Q\d$', parts[i]):
        i += 1

    # On s'arrête immédiatement si le chemin ne contient pas le trimestre
    if i == len(parts):
        return path_infos, False, []

    # On détermine l'index du dossier représentant le segment (qui doit être le dossier parent du fichier)
    last_part_index = i

    # Le premier dossier est donc nommé d'après le trimestre
    path_infos.trimestre = parts[i]

    # Le dossier suivant est nommé d'après le genre
    try:
        path_infos.genre = Genre[parts[i + 1]] if i + 1 < len(parts) else None
    except KeyError:
        path_infos.genre = None

    # Vérification si le chemin contient un dossier de travail ou une collection
    try:
        if i + 3 < len(parts) and parts[i + 3] in [e.name for e in CodeDossier]:  # Animation HORS d'une collection
            path_infos.animation = parts[i + 2] if i + 2 < len(parts) else None
            path_infos.code_dossier = CodeDossier[parts[i + 3]] if i + 3 < len(parts) else None

            # On vérifie si le chemin contient la phase après le code du dossier
            if i + 4 < len(parts) and parts[i + 4] in [e.name for e in Phase]:
                path_infos.phase = Phase[parts[i + 4]] if i + 4 < len(parts) else None
                path_infos.segment = parts[i + 5] if i + 5 < len(parts) else None
                last_part_index += 5
            else:
                path_infos.segment = parts[i + 4] if i + 4 < len(parts) else None
                last_part_index += 4
        else:  # Animation DANS une collection
            path_infos.collection = parts[i + 2] if i + 2 < len(parts) else None
            path_infos.animation = parts[i + 3] if i + 3 < len(parts) else None

            if i + 4 < len(parts) and parts[i + 4] in [e.name for e in CodeDossier]:
                path_infos.code_dossier = CodeDossier[parts[i + 4]]

            # On vérifie si le chemin contient la phase après le code du dossier
            if i + 5 < len(parts) and parts[i + 5] in [e.name for e in Phase]:
                path_infos.phase = Phase[parts[i + 5]] if i + 5 < len(parts) else None
                path_infos.segment = parts[i + 6] if i + 6 < len(parts) else None
                last_part_index += 6
            else:
                # Actuellement n'importe quelle chaîne de caractères est prise pour un segment
                path_infos.segment = parts[i + 5] if i + 5 < len(parts) else None
                last_part_index += 5
    except (IndexError, KeyError):
        pass

    # On vérifie que le dossier représentant le segment soit bien le dossier parent du fichier
    if last_part_index != len(parts) - 2:
        return path_infos, False, ["Le dossier parent ne représente pas le segment (ex : SLG, MARO, etc.)."]

    # La dernière partie du chemin est le nom du fichier
    filename = parts[len(parts) - 1]
    basename, _ = extract_basename_and_extension(filename)

    prefix = path_infos.trimestre + "-"
    if path_infos.collection:
        prefix += path_infos.collection + "-"

    # Le nom du fichier doit être correctement prefixé : TRIMESTRE-[COLLECTION-]ANIMATION-
    prefix += path_infos.animation + "-"

    if not basename.startswith(prefix):
        return path_infos, False, [f"Le nom du fichier doit commencer par : {prefix}"]

    # La partie restante doit suivre un format spécifique : NOM-PRODUIT-(VERSION)-DATE
    suffix = basename[len(prefix):]

    # Exemple : MOTIF-SMALL-500X500-(02)-250320
    regex = r'^([A-Z0-9-]+)(?:-(\d+x\d+))?-\((\d+)\)-(\d{6})$'
    match = re.match(regex, suffix)

    if not match:
        return path_infos, False, [f"La fin du nom du fichier n'a pas le format attendu : {suffix}"]

    produit, dimensions, version, date = match.groups()
    path_infos.produit = produit

    if dimensions:
        path_infos.produit += "-" + dimensions  # On considère que les dimensions font partie du nom du produit

    path_infos.dimensions = dimensions
    path_infos.version = int(version)
    path_infos.date = date

    return path_infos, True, []

In [7]:
test_paths = [
    ["/Volumes/Public/TRANSFORMATIONS/24-Q4/HOMME/MNG-TRANSPARENT/TRA/TRAVEL/24-Q4-MNG-TRANSPARENT-POCHETTE-TRIO-(03)-240108.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-TRAVEL-BAG-48H-LAGOON-(01)-231122.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-POCHETTE-COSMETIQUE-(05)-240208.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-POCHETTE-COSMETIQUE-(02)-231207.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-KEEPALL-45-(03)-240208.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/TRAVEL/24-Q3-RESORT-W-24-CARIBBEAN-HORIZON-55-(02)-231220.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-WALLET-FRAMBOISE-(01)-240104.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-WALLET-(011)-231221.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-ZIPPY-COIN-PURSE-(04)-231221.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q3/FEMME/RESORT-W-24/CARIBBEAN/TRA/SLG/24-Q3-RESORT-W-24-CARIBBEAN-PORTE-CARTES-SIMPLE-TURQUOISE-(04)-240111.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/LG/25-Q1-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/DUPLI/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/PROD/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", True, []],
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA_/PROD/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    # Ci-dessous, on aimerait plutôt que `PROD__` ne soit pas pris pour le nom du segment. Mais aucune vérification n'est faite pour l'instant.
    ["/Users/louis/LV/00-Q0/HOMME/ANIMATION/TRA/PROD__/LG/00-Q0-ANIMATION-SPEEDY-25-(01)-240923.ai", False, []],
    ["/Users/louis/LV", False, []],
    ["00-Q0", False, []],
    ["", False, []],
    # ["/Volumes/Public/TRANSFORMATIONS/25-Q2/FEMME/MON-MNG/TRA/MARO/25-Q2-MON-MNG-ON-THE-GO-GM-MNG-(014)-241011.ai", False, []],
    # ["/Volumes/public/TRANSFORMATIONS/23-Q1/HOMME/LADY-B-M/INFINITY-DOTS/LIV/BELT/23-Q1-LVxYK-M-INFINITY-DOTS-BELT-40-(03)-220802.dxf", True, []],
    ["/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/ARCHIVES/abc/24-Q1-SHOW-M-SS24-DAMIER-POP-PYRAMIDE-(04)-250206.ai", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/ARCHIVES/24-Q1-SHOW-M-SS24-DAMIER-POP-PYRAMIDE-(03)-250206.ai", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG", False, []],
    ["/Volumes/Public/TRANSFORMATIONS/24-Q1/HOMME/SHOW-M-SS24/DAMIER-POP/TRA/SLG/24-Q1-SHOW-M-SS24-DAMIER-POP-MODELE-(01)-250206.ai", True, []],
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/MNG-SEQUINS/TRA/SLG/25Q4-MNG-SEQUINS-POCHETTE-ACCESSOIRES-(02)-250227.ai", False, []], # Manque un tiret dans le trimestre.
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/X-MAS/WINTERY-TRAVEL/TRA/SLG/25-Q4-X-MAS-WINTERY-TRAVEL-PFLISAaaaa-(03)-250129.ai", False, []], # Minuscules dans le nom du produit.
    ["/Volumes/Public/TRANSFORMATIONS/25-Q4/FEMME/X-MAS/WINTERY-TRAVEL/TRA/SLG/25-Q4-X-MAS-WINTERY-TRAVEL-PFLISAAAA-(03)-250129.ai", True, []],
    ["/Volumes/public/TRANSFORMATIONS/PLACEMENT/26-Q1/FEMME/CLASH-SS/BLUE-FLOWER-IKAT/TRA/DUPLI/MOTIF/26-Q1-CLASH-SS-BLUE-FLOWER-IKAT-MOTIF-SMALL-500X500-(02)-250320.ai", True, []], # Valide mais le `X` devrait être minuscule : ici le nom du produit identifié est `"MOTIF-SMALL-500X500`.
    ["/Volumes/public/TRANSFORMATIONS/PLACEMENT/26-Q1/FEMME/CLASH-SS/BLUE-FLOWER-IKAT/TRA/DUPLI/MOTIF/26-Q1-CLASH-SS-BLUE-FLOWER-IKAT-MOTIF-SMALL-500x500-(02)-250320.ai", True, []], # Ajout des éventuelles dimensions du produit.
]

for path, expected_return, _ in test_paths:
    path_infos, path_is_valid, error_messages = parse_path(path)
    if path_is_valid != expected_return:
        print(path_infos)
        print(path)
        if error_messages:
            for error_message in error_messages:
                print(f"- {error_message}")

In [9]:
print(CodeDossier.LIV.value)

LIV
